# OmniVoice - Deploy Colab test nhanh
Notebook này giúp bạn chạy backend FastAPI trên Colab để test queue/concurrency.

**Flow:**
1. Cài dependency
2. Clone repo
3. Set env (bao gồm `OMNIVOICE_MAX_CONCURRENT_LONG_JOBS`)
4. Run backend
5. Expose public URL qua ngrok


In [ ]:
!nvidia-smi || true
!apt-get -y update
!apt-get -y install ffmpeg
!pip install -U pip setuptools wheel

In [ ]:
import os

# TODO: đổi thành repo của bạn
REPO_URL = "https://github.com/your-org/OmniVoice-Space.git"
REPO_DIR = "/content/OmniVoice-Space"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    print("Repo đã tồn tại, dùng lại thư mục cũ")

%cd /content/OmniVoice-Space
!git pull --rebase || true

In [ ]:
%cd /content/OmniVoice-Space
!pip install -r requirements.txt
!pip install fastapi uvicorn pyngrok nest_asyncio python-multipart

In [ ]:
import os
import torch

# Tự chọn device theo runtime Colab
os.environ['OMNIVOICE_DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['OMNIVOICE_DTYPE'] = 'auto'

# Bạn đang muốn test concurrency
os.environ['OMNIVOICE_MAX_CONCURRENT_LONG_JOBS'] = '2'

print('OMNIVOICE_DEVICE =', os.environ['OMNIVOICE_DEVICE'])
print('OMNIVOICE_DTYPE =', os.environ['OMNIVOICE_DTYPE'])
print('OMNIVOICE_MAX_CONCURRENT_LONG_JOBS =', os.environ['OMNIVOICE_MAX_CONCURRENT_LONG_JOBS'])

In [ ]:
%cd /content/OmniVoice-Space
import subprocess, sys

# Chạy backend trong background
cmd = [sys.executable, '-m', 'uvicorn', 'server.api:app', '--host', '0.0.0.0', '--port', '8000']
backend_proc = subprocess.Popen(cmd)
print('Backend PID:', backend_proc.pid)

In [ ]:
import time
import requests
from pyngrok import ngrok

time.sleep(3)

# Nếu bạn có token ngrok riêng, set trước khi mở tunnel:
# ngrok.set_auth_token('YOUR_NGROK_AUTH_TOKEN')

public_url = ngrok.connect(8000, bind_tls=True).public_url
print('Public URL:', public_url)

r = requests.get('http://127.0.0.1:8000/api/health', timeout=20)
print('Health:', r.status_code, r.json())
print('=> Base API URL để test:', public_url)

## Gợi ý test nhanh
- `GET {public_url}/api/health`
- `GET {public_url}/api/jobs`
- `POST {public_url}/api/jobs/long` với form-data `script_text` để tạo job

Bạn có thể dùng Postman hoặc frontend local (trỏ API qua URL ngrok).

In [ ]:
# Dừng backend khi test xong
try:
    backend_proc.terminate()
    backend_proc.wait(timeout=10)
    print('Đã dừng backend')
except Exception as e:
    print('Không dừng được sạch, error =', e)